<a href="https://colab.research.google.com/github/OnwutaKelvin/ML-notebooks/blob/main/miniResN_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Importing Libraries
import torch
import torch.nn as nn

In [ ]:
#Residual Block
class ResidualBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()

    self.conv1 = nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False
    )
    self.bn1 = nn.BatchNorm2d(out_channels)

    self.conv2 = nn.Conv2d(
        out_channels,
        out_channels,
        kernel_size=3,
        padding=1,
        bias=False
    )
    self.bn2 = nn.BatchNorm2d(out_channels)

    self.shortcut = nn.Sequential()

    if stride != 1 or in_channels != out_channels:
      self.shortcut = nn.Sequential(
          nn.Conv2d(
              in_channels,
              out_channels,
              kernel_size=3,
              stride=stride,
              padding=1,
              bias=False
         ),
         nn.BatchNorm2d(out_channels)
      )

  def forward(self, x):
    identity = self.shortcut(x)

    out = self.conv1(x)
    out = self.bn1(out)
    out = torch.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    out += identity
    out = torch.relu(out)

    return(out)

In [ ]:
#Building the mini ResNet
class MiniResNet(nn.Module):
  def __init__(self, num_classes):
    super().__init__()

    self.stem = nn.Sequential(
        nn.Conv2d(
            3,
            32,
            kernel_size=3,
            padding=1,
            bias=False
        ),
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True)
    )

    self.layer1 = ResidualBlock(
        32,
        32,
        stride=1
    )

    self.layer2 = ResidualBlock(
        32,
        64,
        stride=2
    )

    self.layer3 = ResidualBlock(
        64,
        128,
        stride=2
    )

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    self.fc = nn.Linear(
        128,
        num_classes
    )

  def forward(self, x):
    x = self.stem(x)

    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)

    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.fc(x)

    return(x)

In [ ]:
#Data Transformation and Augmentation
from torchvision import datasets
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomCrop(
        32,
        padding=4,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        saturation=0.2,
        contrast=0.2,
        brightness=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

In [ ]:
#Datasets

train_dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform
)


In [ ]:
#Data Loading
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

In [ ]:
#Model
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(device)

model = MiniResNet(num_classes=10).to(device)

cuda


In [ ]:
#Loss and Optimizer
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
#Training Loop
from tqdm import tqdm

epochs = 20

best_acc = 0

for epoch in range(epochs):

  model.train()

  running_loss = 0

  loop = tqdm(train_loader)

  for images, labels in loop:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = loss_fn(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

    loop.set_description(
        f"Epoch: [{epoch+1}/{epochs}]"
    )

    loop.set_postfix(
        loss=loss.item()
    )

  #Evaluation
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for images, label in test_loader:
      images = images.to(device)
      labels = label.to(device) # Corrected: using 'label' from the test_loader iteration

      outputs = model(images)
      predicted = outputs.argmax(1)

      correct += (predicted == labels).sum().item()
      total += labels.size(0)

accuracy = 100 * correct / total

print(
    f"Test Accuracy: {accuracy:.2f}%"
)

if accuracy > best_acc:
  best_acc = accuracy

  torch.save(
      model.state_dict(),
      "best_model.pth"
  )

Epoch: [20/20]: 100%|██████████| 391/391 [00:34<00:00, 11.24it/s, loss=0.231]


Test Accuracy: 83.64%
